In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv


# Enron received-mail sampling for a junk-mail filter
Goal: 500+ labelled received emails. Source: wcukierski/enron-email-dataset. Kaggle: rehanmhate

## Step 0 – Find the data path

In [3]:
import os
for root, _, files in os.walk('/kaggle/input'):
    print(root, files)

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/wcukierski []
/kaggle/input/datasets/wcukierski/enron-email-dataset ['emails.csv']


## Step 1 – Load raw emails

In [4]:
import pandas as pd, email
PATH = '/kaggle/input/datasets/wcukierski/enron-email-dataset/emails.csv'
df = pd.read_csv(PATH)
print(df.shape); df.head(3)

(517401, 2)


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...


## Step 2 – Build key, mailbox, folder; drop sent folders

In [5]:
df['key'] = df['file'].str.rstrip('.')          # e.g. allen-p/inbox/12
parts = df['key'].str.split('/')
df['mailbox'] = parts.str[0]
df['folder'] = parts.str[1:-1].str.join('/')

SENT = {'sent', 'sent_items', '_sent_mail'}
df['is_sent'] = parts.apply(lambda p: any(x in SENT for x in p[1:-1]))
print(df['is_sent'].value_counts())

recv = df[~df['is_sent']].copy()
print(len(recv), 'received emails |', recv['mailbox'].nunique(), 'mailboxes')

is_sent
False    391075
True     126326
Name: count, dtype: int64
391075 received emails | 150 mailboxes


## Step 3 – Parse subject, from, to, body

In [6]:
def parse(raw):
    m = email.message_from_string(raw)
    b = m.get_payload()
    return pd.Series({
        'subject': m.get('Subject', '') or '',
        'from_address': m.get('From', '') or '',
        'to_addresses': m.get('To', '') or '',
        'body': b if isinstance(b, str) else '',
    })

recv = recv.join(recv['message'].apply(parse))
print(recv.shape)
recv[['key', 'subject', 'from_address', 'to_addresses']].head()

(391075, 10)


,key,subject,from_address,to_addresses
602,allen-p/all_documents/1,"December 14, 2000 - Bear Stearns' predictions ...",1.11913372.-2@multexinvestornetwork.com,pallen@enron.com
603,allen-p/all_documents/10,Bloomberg Power Lines Report,messenger@ecm.bloomberg.com,
604,allen-p/all_documents/100,Consolidated positions: Issues & To Do list,phillip.allen@enron.com,keith.holst@enron.com
605,allen-p/all_documents/101,Consolidated positions: Issues & To Do list,phillip.allen@enron.com,keith.holst@enron.com
606,allen-p/all_documents/102,,phillip.allen@enron.com,david.delainey@enron.com


## Step 4 – Spread across folders and mailboxes

In [7]:
print('Top folders:')
print(recv['folder'].value_counts().head(25))
print('\nEmails per mailbox:')
print(recv.groupby('mailbox').size().describe())

Top folders:
folder
all_documents                            128103
discussion_threads                        58609
deleted_items                             50991
inbox                                     41509
notes_inbox                               36665
calendar/untitled                          4478
archiving/untitled                         4477
personal                                   2420
attachments                                2026
meetings                                   1872
calendar                                   1655
schedule_crawler                           1398
logistics                                  1192
tw_commercial_group                        1159
california                                 1052
bill_williams_iii                          1022
deal_communication/deal_discrepancies       897
_americas/esvl                              715
e_mail_bin                                  614
tufco                                       609
resumes             